# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [7]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 63 packages in 276ms                                        
Prepared 2 packages in 54ms                                                  typer                ------------------------------ 120.42 KiB/120.42 KiB       typer                ------------------------------ 14.77 KiB/120.42 KiB        
Uninstalled 2 packages in 20ms
Installed 2 packages in 2ms                                 
 - opentelemetry-api==1.45.0
 + opentelemetry-api==1.45.1
 - typer==0.27.2
 + typer==0.27.3


## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [8]:
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import requests


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [9]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 02334b1d


## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [10]:
persona = (
    "Professoressa Giulia, a warm and lively Italian teacher who sprinkles short Italian phrases "
    "into her replies (always with an English translation), celebrates right answers with "
    "\"Bravissimo!\", and corrects mistakes gently by showing the correct form and the rule behind it."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text


# Module 5 tool: log every missed question to a review file
def add_to_ripasso_tool(question):
    """Append a missed question to ripasso.txt ("ripasso" is Italian for "review")."""
    print(f"Adding to your ripasso list: {question}")
    with open("ripasso.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return f"Saved to ripasso.txt for later review: {question}"


print(persona)
print(notes[:300], "...")

incorrect startxref pointer(1)
parsing for Object Streams


Professoressa Giulia, a warm and lively Italian teacher who sprinkles short Italian phrases into her replies (always with an English translation), celebrates right answers with "Bravissimo!", and corrects mistakes gently by showing the correct form and the rule behind it.
Italian Present Tense (Il Presente Indicativo)
The present tense in Italian describes what is happening now, habits, and near-future plans. Italian
verbs are grouped by their infinitive ending: -are, -ere, or -ire. To conjugate a regular verb, drop the
ending to get the stem, then add the endings fo ...


In [12]:
add_to_ripasso_tool_json = {
    "name": "add_to_ripasso_tool",
    "description": (
        "Save a quiz question the student answered incorrectly to their ripasso (review) list. "
        "Call this every time the student gets a question wrong, once per missed question."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The full text of the question the student got wrong"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [13]:
def translate_word_tool(word):
    """Look up the English meaning of an Italian word or short phrase using the free MyMemory API."""
    print(f"Looking up the meaning of: {word}")
    try:
        response = requests.get(
            "https://api.mymemory.translated.net/get",
            params={"q": word, "langpair": "it|en"},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException:
        return f"The dictionary is not available right now, so I could not look up '{word}'."

    translation = data.get("responseData", {}).get("translatedText", "")
    if data.get("responseStatus") != 200 or not translation:
        return f"I could not find a translation for '{word}' right now."

    return f"'{word}' means '{translation}' in English."

In [14]:
translate_word_tool("mangiare")  # quick manual test of your API-backed tool

Looking up the meaning of: mangiare


"'mangiare' means 'to eat' in English."

## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [15]:
def track_score_tool(correct):
    """Keep a running tally of right and wrong answers in score.json and report the total."""
    try:
        with open("score.json", "r", encoding="utf-8") as f:
            score = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        score = {"correct": 0, "wrong": 0}

    if correct:
        score["correct"] += 1
    else:
        score["wrong"] += 1

    with open("score.json", "w", encoding="utf-8") as f:
        json.dump(score, f)

    total = score["correct"] + score["wrong"]
    print(f"Score updated: {score['correct']} right, {score['wrong']} wrong")
    return f"Score so far: {score['correct']} out of {total} correct."

In [16]:
track_score_tool(True)  # quick manual test of your third tool

Score updated: 1 right, 0 wrong


'Score so far: 1 out of 1 correct.'

## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [18]:
translate_word_tool_json = {
    "name": "translate_word_tool",
    "description": (
        "Look up the English meaning of an Italian word or short phrase. Call this when the student "
        "asks what an Italian word means, or asks for a vocabulary hint during a question."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "word": {
                "type": "string",
                "description": "The Italian word or short phrase to translate into English, for example 'mangiare'"
            }
        },
        "required": ["word"],
        "additionalProperties": False
    }
}

track_score_tool_json = {
    "name": "track_score_tool",
    "description": (
        "Update the student's running score. Call this once after every quiz answer you grade, "
        "with correct set to true for a right answer and false for a wrong one."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "correct": {
                "type": "boolean",
                "description": "True if the student's answer was correct, false if it was wrong"
            }
        },
        "required": ["correct"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": add_to_ripasso_tool_json},
    {"type": "function", "function": translate_word_tool_json},
    {"type": "function", "function": track_score_tool_json},
]

## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [ ]:
system_prompt = f"""
# TODO: write your system prompt here, using {persona} and {notes}, with one clear rule
# sentence per tool telling the model when to use it
"""

display(Markdown(system_prompt))

## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [ ]:
def call_tool(name, arguments):
    # TODO: if/elif/else on `name`, calling the matching tool function with its argument(s)
    # pulled out of `arguments`, and returning a short string in every branch, including an
    # "Unknown tool" fallback in the else branch
    pass

## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    # TODO: while response.choices[0].finish_reason == "tool_calls":
    #     - message_obj = response.choices[0].message
    #     - messages.append(message_obj)
    #     - for tool_call in message_obj.tool_calls:
    #           - name = tool_call.function.name
    #           - arguments = json.loads(tool_call.function.arguments)
    #           - result = call_tool(name, arguments)
    #           - messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    #     - response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.